# Naive Bayes Classifier Implementation


---


### 01. Library Installation


In [ ]:
%pip install -qq imbalanced-learn matplotlib numpy pandas scikit-learn seaborn


### 02. Library Imports


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from imblearn.over_sampling import RandomOverSampler

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler


### 03. Data Loading and Preprocessing


Looking at the dataset that will be processed, this is the **Skin Segmentation Dataset** which contains RGB pixel data collected for the purpose of skin detection in computer vision applications. The dataset is used for binary classification to distinguish between skin and non-skin pixels.

**Dataset characteristics:**
- **Total samples**: 245,057 observations
- **Features**: 3 continuous variables representing RGB color channels
- **Target**: Binary classification (Skin vs Non-skin pixels)
    - Class 1: Non-skin pixels (background/objects)
    - Class 2: Skin pixels (human skin) → mapped to 1
    - Class 1: Non-skin pixels → mapped to 0

**Feature descriptions:**
- `B`: Blue color channel intensity (0-255)
- `G`: Green color channel intensity (0-255)
- `R`: Red color channel intensity (0-255)

**Dataset context:**
This dataset was originally created for skin detection research in computer vision and image processing. Skin detection is a crucial preprocessing step in many applications including:
- Face detection and recognition systems
- Hand gesture recognition
- Content filtering systems
- Human-computer interaction
- Medical image analysis
- Surveillance systems

**Class distribution:**
- Non-skin pixels: 194,198 samples (~79.2%)
- Skin pixels: 50,859 samples (~20.8%)

The dataset shows a significant class imbalance, with non-skin pixels being approximately 4 times more frequent than skin pixels. This imbalance reflects real-world scenarios where skin pixels typically represent a smaller portion of natural images.

**Data collection:**
The RGB values were collected from face images of people of different ages, races, and genders under various lighting conditions and backgrounds. This diversity ensures the model can generalize well across different scenarios.

In [ ]:
# Importing and checking the dataframe

dataframe = pd.read_csv('../../datasets/skin_segmentation/Skin_NonSkin.txt', sep = '\t', header = None)
dataframe.head()


In [ ]:
# Renaming columns accordingly to the dataset documentation

columns_name = ['B', 'G', 'R', 'y']
dataframe.columns = columns_name
dataframe.head()


In [ ]:
# Checking the possible values for the target variable

dataframe['y'].value_counts()


In [ ]:
# Changing the target variable to binary values

dataframe['y'] = dataframe['y'].map({2: 0, 1: 1})
dataframe['y'].value_counts()


### 04. Data Visualization


In [ ]:
for label in dataframe:
    if label == 'y' or dataframe[label].dtype == 'object':
        continue

    plt.figure(figsize = (10, 6))

    for class_value in dataframe['y'].unique():
        subset = dataframe[dataframe['y'] == class_value]
        sns.kdeplot(subset[label], fill = True, alpha = 0.5, label = f'Class {class_value}')

    plt.title(f'Distribution of {label} by Class')
    plt.xlabel(label)
    plt.ylabel('Probability')
    plt.legend(title = 'Class')
    plt.grid()

    plt.show()


### 05. Dataset Splitting and Scaling


In [ ]:
# Shuffling the dataframe

dataframe = dataframe.sample(frac = 1, random_state = 42).reset_index(drop = True)


In [ ]:
dataframe.info()


In [ ]:
dataframe.head()


In [ ]:
# Defining the train, validation and test datasets sizes

train_size = 0.7
validation_size = 0.15
test_size = 0.15


In [ ]:
# Defining the train, validation and test datasets

train_dataset = dataframe[:int(train_size * len(dataframe))]
validation_dataset = dataframe[int(train_size * len(dataframe)):int((train_size + validation_size) * len(dataframe))]
test_dataset = dataframe[int((train_size + validation_size) * len(dataframe)):]

print('\nDatasets sizes before scaling and oversampling:')
print(f'Train dataset size: {len(train_dataset)} samples')
print(f'Validation dataset size: {len(validation_dataset)} samples')
print(f'Test dataset size: {len(test_dataset)} samples')


In [ ]:
# Defining a function to scale the data and oversample if necessary

def scale_data(dataframe, oversample = False):
    features = dataframe.columns[:-1]
    target = dataframe.columns[-1]

    X = dataframe[features].values
    y = dataframe[target].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    if oversample:
        ros = RandomOverSampler(random_state = 42)
        X_resampled, y_resampled = ros.fit_resample(X_scaled, y)
        dataframe_scaled = np.hstack((X_resampled, np.reshape(y_resampled, (-1, 1))))
        return dataframe_scaled, X_resampled, y_resampled
    else:
        dataframe_scaled = np.hstack((X_scaled, np.reshape(y, (-1, 1))))
        return dataframe_scaled, X_scaled, y


In [ ]:
train_dataset, X_train, y_train = scale_data(train_dataset, oversample = True)
validation_dataset, X_validation, y_validation = scale_data(validation_dataset, oversample = False)
test_dataset, X_test, y_test = scale_data(test_dataset, oversample = False)

print('\nDatasets sizes after scaling and oversampling:')
print(f'Train dataset size: {len(train_dataset)} samples')
print(f'Validation dataset size: {len(validation_dataset)} samples')
print(f'Test dataset size: {len(test_dataset)} samples')


### 06. Naive Bayes Implementation and Evaluation


In [ ]:
# Naive Bayes implementation

naive_bayes_model = GaussianNB()
naive_bayes_model.fit(X_train, y_train)

y_predictions_validation = naive_bayes_model.predict(X_validation)
y_predictions_test = naive_bayes_model.predict(X_test)

print('Validation Set Classification Report:')
print(classification_report(y_validation, y_predictions_validation))

print('Test Set Classification Report:')
print(classification_report(y_test, y_predictions_test))
